In [ ]:
import os
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np
from xtheta.data.adapters.hensen import load_hensen_dataset
from xtheta.data.bell_chsh import compute_chsh_variants
from xtheta.data.schema import BellEventSchema

# 06 Hensen (Delft) 2015 Open-Data Audit

This notebook audits the raw Hensen et al. (2015) data and our internal adapter mapping to ensure CHSH S calculation accuracy.

**Scientific Warning:**
Phi_eff is an effective phenomenological parameter only. Without gravitational path, altitude, curvature, or spacetime-baseline metadata, the open Bell/CHSH datasets are not evidence of spacetime-induced X-Theta holonomy. They validate the computational mapping from observed CHSH statistics to effective X-Theta parameters.

## 1. Manual Raw Audit

We inspect the raw file structure directly.

In [ ]:
raw_path = Path("../data/open_bell/hensen/raw/bell_open_data.txt")

if not raw_path.exists():
    print(f"[ERROR] Raw data not found at {raw_path}")
    print("Please run: python ../scripts/download_open_data.py --dataset hensen")
else:
    print(f"Raw file path: {raw_path.resolve()}")
    raw_lines = raw_path.read_text(encoding='utf-8').splitlines()
    print(f"\nFirst 10 raw lines:")
    for line in raw_lines[:10]:
        print(line)

## 2. Adapter Audit

We use the `load_hensen_dataset` adapter to parse the data and verify mapping.

In [ ]:
if raw_path.exists():
    # Load all data
    df_iter = load_hensen_dataset(str(raw_path))
    df = pd.concat(list(df_iter), ignore_index=True)
    
    print(f"Parsed DataFrame Preview (first 5 rows):")
    display(df.head())
    
    schema = BellEventSchema()
    
    print("\nUnique Alice Settings:", df[schema.alice_setting].unique())
    print("Unique Bob Settings:", df[schema.bob_setting].unique())
    
    print("\nAlice Outcome Counts:")
    print(df[schema.alice_outcome].value_counts())
    
    print("\nBob Outcome Counts:")
    print(df[schema.bob_outcome].value_counts())
    
    print("\nSetting-pair counts:")
    counts = df.groupby([schema.alice_setting, schema.bob_setting]).size()
    print(counts)

## 3. Correlation and CHSH Audit

Calculate expectations $E(a,b)$ and CHSH variants.

In [ ]:
if raw_path.exists():
    df['ab'] = df[schema.alice_outcome] * df[schema.bob_outcome]
    
    # Calculate E(a,b)
    expectations = df.groupby([schema.alice_setting, schema.bob_setting])['ab'].mean()
    
    E00 = expectations.get((0, 0), 0.0)
    E01 = expectations.get((0, 1), 0.0)
    E10 = expectations.get((1, 0), 0.0)
    E11 = expectations.get((1, 1), 0.0)
    
    print(f"E00: {E00:.4f}")
    print(f"E01: {E01:.4f}")
    print(f"E10: {E10:.4f}")
    print(f"E11: {E11:.4f}")

    variants = compute_chsh_variants(E00, E01, E10, E11)
    
    print("\nCHSH Sign Variants:")
    for k, v in variants.items():
        if k not in ['max_abs', 'max_abs_convention']:
            print(f"  {k}: {v:.4f}")
            
    print(f"\nMax Absolute CHSH: {variants['max_abs']:.4f} (Convention: {variants['max_abs_convention']})")